# Run9 Alpha Training
Thin orchestration notebook for the canonical Run9 alpha-generation training path. Durable logic lives in `src/config.py`, `src/notebook_helpers/tcn_phase1.py`, and the agent/environment source files.


## 1) Colab Setup
Clone/sync the repo, clean previous outputs, install requirements, and verify GPU availability.


In [ ]:
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

TRAIN_REPO_URL = "https://github.com/Dave-DKings/tcn_tape_vectorized_version.git"
TRAIN_REPO_DIR = Path("/content/tcn_tape_vectorized_version_clean")
TRAIN_BRANCH = "feature/run6-tuning-20260306"

if not (TRAIN_REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', TRAIN_REPO_URL, str(TRAIN_REPO_DIR)], check=True)

subprocess.run(['git', '-C', str(TRAIN_REPO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(TRAIN_REPO_DIR), 'checkout', TRAIN_BRANCH], check=True)
subprocess.run(['git', '-C', str(TRAIN_REPO_DIR), 'reset', '--hard', f'origin/{TRAIN_BRANCH}'], check=True)

purge_paths = [
    TRAIN_REPO_DIR / 'tcn_fusion_results',
    TRAIN_REPO_DIR / 'tcn_results',
    TRAIN_REPO_DIR / 'tcn_att_results',
    TRAIN_REPO_DIR / 'output_log',
    TRAIN_REPO_DIR / 'output_logs',
    TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
    TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
    TRAIN_REPO_DIR / 'data' / 'daily_ohlcv_assets.csv',
    TRAIN_REPO_DIR / 'data' / 'processed_daily_macro_features.csv',
]

for path in purge_paths:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.is_file():
        path.unlink(missing_ok=True)

for cache_dir in TRAIN_REPO_DIR.rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for ckpt_dir in TRAIN_REPO_DIR.rglob('.ipynb_checkpoints'):
    shutil.rmtree(ckpt_dir, ignore_errors=True)

for mod in list(sys.modules.keys()):
    if mod.startswith('src.') or mod.startswith('src_'):
        del sys.modules[mod]
gc.collect()

os.chdir(TRAIN_REPO_DIR)
if str(TRAIN_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_REPO_DIR))

requirements_file = TRAIN_REPO_DIR / 'requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_file)], check=True)

print('[OK] Repo synced:', TRAIN_REPO_DIR)
print('[OK] Branch:', TRAIN_BRANCH)
print('[OK] Requirements installed')


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
if not gpus:
    raise RuntimeError('No GPU visible to TensorFlow. In Colab: Runtime -> Change runtime type -> GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy('float32')
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


## 2) Imports
Import the canonical source helpers and training entrypoints.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import build_run9_alpha_config, assert_run9_alpha_config
from src.csv_logger import CSVLogger
from src.notebook_helpers.tcn_phase1 import prepare_phase1_dataset, run_experiment6_tape

RUN_ID = 'run9'
TRAIN_RANDOM_SEED = 42
ANALYSIS_END_DATE = None


## 3) Build Canonical Run9 Config and Dataset
Create the source-backed Run9 config, assert no drift, and prepare the dataset once.


In [ ]:
train_config = build_run9_alpha_config('phase1', analysis_end_date=ANALYSIS_END_DATE)
assert_run9_alpha_config(train_config)

tp = train_config['training_params']
ap = train_config['agent_params']
ppo = ap['ppo_params']
env = train_config['environment_params']

print('[Run9] Canonical config ready')
print('  split_date =', train_config['TRAIN_TEST_SPLIT_DATE'])
print('  architecture =', ap['actor_critic_type'])
print('  regime_conditioning =', ap['regime_conditioning_enabled'])
print('  distributional_critic =', ap['distributional_critic_enabled'])
print('  cvar_advantage_weight =', ppo['cvar_advantage_weight'])
print('  lagrangian =', {
    'threshold': ppo['lagrangian_cvar_threshold'],
    'lr': ppo['lagrangian_cvar_lr'],
    'lambda_max': ppo['lagrangian_cvar_lambda_max'],
    'penalty_scale': ppo['lagrangian_cvar_penalty_scale'],
})
print('  drawdown =', {
    'target': env['drawdown_constraint']['target'],
    'tolerance': env['drawdown_constraint']['tolerance'],
    'penalty_coef': env['drawdown_constraint']['penalty_coef'],
    'lambda_carry_decay': env['drawdown_constraint']['lambda_carry_decay'],
})
print('  dispersion =', {
    'hhi_coef': ppo['alpha_diversity_coef'],
    'dispersion_coef': ppo['alpha_dispersion_coef'],
    'dispersion_target_std': ppo['alpha_dispersion_target_std'],
})
print('  deterministic_validation =', tp['deterministic_validation_checkpointing_enabled'])

train_phase1_data = prepare_phase1_dataset(
    train_config,
    force_download=True,
    preparation_artifacts_dir=str(TRAIN_REPO_DIR / 'data_exports'),
)

actuarial_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Actuarial_')]
actuarial_non_null = {c: int(train_phase1_data.master_df[c].notna().sum()) for c in actuarial_cols}
if not actuarial_cols or any(v == 0 for v in actuarial_non_null.values()):
    raise RuntimeError(f'Actuarial features missing or empty: {actuarial_non_null}')
fundamental_cols = [c for c in train_phase1_data.master_df.columns if str(c).startswith('Fundamental_')]
if fundamental_cols:
    raise RuntimeError(f'Fundamental columns should be absent: {fundamental_cols}')

print('[OK] Train shape:', train_phase1_data.train_df.shape)
print('[OK] Test shape:', train_phase1_data.test_df.shape)
print('[OK] Actuarial feature check passed:', actuarial_non_null)
print('[OK] Fundamental feature check passed: none present')


## 4) Run Training
Launch the canonical Run9 training path.


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    training_params = train_config['training_params']
    print('[START] Starting training')
    print('Architecture:', train_config['agent_params'].get('actor_critic_type'))
    print('max_total_timesteps:', training_params['max_total_timesteps'])
    print('num_parallel_envs:', training_params.get('num_parallel_envs', 1))

    train_experiment6 = run_experiment6_tape(
        phase1_data=train_phase1_data,
        config=train_config,
        random_seed=TRAIN_RANDOM_SEED,
        csv_logger_cls=CSVLogger,
        use_covariance=True,
        architecture=train_config['agent_params'].get('actor_critic_type'),
        timesteps_per_update=training_params.get('timesteps_per_ppo_update', 1008),
        max_total_timesteps=training_params['max_total_timesteps'],
    )

    print('[OK] Training complete')
    print('checkpoint_prefix:', train_experiment6.checkpoint_path)
else:
    print('[SKIP] RUN_TRAINING=False')


## 5) Inspect Latest Training Logs
Load the latest training CSVs and inspect the current run.


In [ ]:
TRAIN_RESULTS_ROOT = TRAIN_REPO_DIR / 'tcn_fusion_results'
TRAIN_LOGS_DIR = TRAIN_RESULTS_ROOT / 'logs'

episodes_files = sorted(TRAIN_LOGS_DIR.glob('*episodes*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if not episodes_files:
    print(f'No episodes CSV found in {TRAIN_LOGS_DIR}')
else:
    train_episodes_path = episodes_files[0]
    train_episodes_df = pd.read_csv(train_episodes_path)
    print('Episodes file:', train_episodes_path)
    print('Rows:', len(train_episodes_df))
    display(train_episodes_df.tail(20))

step_diag_files = sorted(TRAIN_LOGS_DIR.glob('*step_diagnostics*.csv'), key=lambda p: p.stat().st_mtime, reverse=True)
if step_diag_files:
    step_diag_path = step_diag_files[0]
    step_diag_df = pd.read_csv(step_diag_path)
    print('Step diagnostics file:', step_diag_path)
    print('Rows:', len(step_diag_df))
    display(step_diag_df.tail(20))


## 6) Export Artifacts (Optional)
Zip the latest results and optionally copy them to Google Drive.


In [ ]:
import subprocess

EXPORT_RESULTS_ZIP = False
COPY_TO_DRIVE = False
EXPORT_PATH = Path(f'/content/tcn_tape_vectorized_{RUN_ID}.zip')

if EXPORT_RESULTS_ZIP:
    include_paths = [
        TRAIN_REPO_DIR / 'tcn_fusion_results',
        TRAIN_REPO_DIR / 'data' / 'phase1_preparation_artifacts',
        TRAIN_REPO_DIR / 'data' / 'master_features_NORMALIZED.csv',
        TRAIN_REPO_DIR / 'data_exports',
        TRAIN_REPO_DIR / 'output_log',
        TRAIN_REPO_DIR / 'output_logs',
    ]
    existing = []
    seen = set()
    for path in include_paths:
        resolved = path.resolve()
        if resolved.exists() and resolved not in seen:
            seen.add(resolved)
            existing.append(resolved)

    if existing:
        if EXPORT_PATH.exists():
            EXPORT_PATH.unlink()
        relative_items = [str(path.relative_to(TRAIN_REPO_DIR)) for path in existing]
        subprocess.run(
            ['bash', '-lc', 'cd "{}" && zip -qr "{}" {}'.format(TRAIN_REPO_DIR, EXPORT_PATH, ' '.join(f'"{item}"' for item in relative_items))],
            check=True,
        )
        print('[OK] Created:', EXPORT_PATH)

        if COPY_TO_DRIVE:
            from google.colab import drive
            drive.mount('/content/drive')
            subprocess.run(['cp', str(EXPORT_PATH), '/content/drive/MyDrive/'], check=True)
            print('[OK] Copied to Drive:', f'/content/drive/MyDrive/{EXPORT_PATH.name}')
    else:
        print('[WARN] Nothing to export.')
else:
    print('[SKIP] EXPORT_RESULTS_ZIP=False')
